# Argus AI — Phase 1: YOLOv8 Fire/Smoke Detection
Dataset: D-Fire (via Kaggle - sayedgamal99/smoke-fire-detection-yolo)

Run all cells top to bottom. Use GPU runtime: Runtime > Change runtime type > T4 GPU

In [ ]:
# 1. Install dependencies
!pip install ultralytics kagglehub -q

In [ ]:
# 2. Kaggle API setup
# Go to kaggle.com -> Account -> Create New API Token -> downloads kaggle.json
# Upload it here:
from google.colab import files
uploaded = files.upload()  # select kaggle.json when prompted

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
# 3. Download dataset
import kagglehub
path = kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo")
print("Path to dataset files:", path)

In [ ]:
# 4. Inspect dataset structure - find train/val folders and any existing data.yaml
!find {path} -maxdepth 3 -type d
print("---")
!find {path} -name "*.yaml"

In [ ]:
# 5. Create data.yaml for YOLOv8 training
# NOTE: after running cell 4, confirm the exact train/images and val/images paths
# and adjust below if folder names differ (e.g. 'train/images' vs 'images/train').

data_yaml = f"""
train: {path}/train/images
val: {path}/val/images

nc: 2
names: ['smoke', 'fire']
"""

with open('/content/data.yaml', 'w') as f:
    f.write(data_yaml)

print(data_yaml)

In [ ]:
# 6. Train YOLOv8
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # nano - fast, good enough for this task. use yolov8s.pt for slightly better accuracy, slower

results = model.train(
    data='/content/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,          # early stopping if no improvement
    project='argus_fire_detection',
    name='yolov8n_dfire',
    device=0               # GPU
)

In [ ]:
# 7. Validate — check mAP, precision, recall
metrics = model.val()
print(metrics)

In [ ]:
# 8. Quick inference test on a validation image
import glob
test_img = glob.glob(f'{path}/val/images/*')[0]
results = model(test_img)
results[0].show()
results[0].save(filename='/content/test_prediction.jpg')

In [ ]:
# 9. Export best weights for download
# Best weights are saved at: argus_fire_detection/yolov8n_dfire/weights/best.pt
from google.colab import files
files.download('argus_fire_detection/yolov8n_dfire/weights/best.pt')

## Next steps
1. Download `best.pt` from cell 9
2. Move it into your local repo: `argus-ai/models/fire_detection/best.pt`
3. Write a local inference test script (FastAPI endpoint later) to load this weight and run predictions on your laptop — inference only, no training needed locally
4. Note down final mAP50, precision, recall from cell 7 — you'll need these numbers for your resume/README
5. Move to Phase 2: flood classifier (U-Net/ResNet)